# Google ADK 2.0 Graph-Based Agentic AI Tutorial

This notebook is a hands-on tutorial for building progressively advanced agentic AI applications using **Google Agent Development Kit (ADK) 2.0 graph-based workflows** with **Gemini**.

## What you will build

1. Single agent
2. Single agent with tools
3. Single agent with tools and Human-in-the-Loop (HITL)
4. Multi-agent supervisor pattern
5. RAG solution using ChromaDB
6. Corrective RAG using ADK graph-based workflow

## Tools used

- Google Gemini LLM through `GOOGLE_API_KEY` from `.env`
- OpenWeatherMap tool
- Tavily Search tool
- Wikipedia tool
- Yahoo Finance news tool through `yfinance`
- ChromaDB vector store

## Important note

ADK 2.0 graph-based workflows are in beta. The notebook uses the current public documentation pattern:

```python
from google.adk import Agent, Workflow, Event
```

Graph-based workflows define execution nodes and edges. Nodes can be agents, tools, custom functions, or human input nodes.

## 0. Environment setup

Create a `.env` file in the same folder as this notebook.

Minimum required:

```env
GOOGLE_API_KEY=your_google_gemini_api_key
```

Recommended optional keys:

```env
OPENWEATHERMAP_API_KEY=your_openweathermap_key
TAVILY_API_KEY=your_tavily_key
```

If optional keys are missing, the related tool functions return a helpful message instead of failing.

In [1]:
# Install dependencies.
# ADK 2.0 is currently pre-GA, so --pre is used.

%pip install -q --upgrade google-adk --pre
%pip install -q python-dotenv requests tavily-python wikipedia yfinance chromadb google-genai pydantic

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-exporter-otlp-proto-grpc 1.41.1 requires opentelemetry-exporter-otlp-proto-common==1.41.1, but you have opentelemetry-exporter-otlp-proto-common 1.38.0 which is incompatible.
opentelemetry-exporter-otlp-proto-grpc 1.41.1 requires opentelemetry-proto==1.41.1, but you have opentelemetry-proto 1.38.0 which is incompatible.
opentelemetry-exporter-otlp-proto-grpc 1.41.1 requires opentelemetry-sdk~=1.41.1, but you have opentelemetry-sdk 1.38.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have open

In [2]:
import os
import json
import textwrap
from typing import Any, Dict, List, Optional
from dotenv import load_dotenv

load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
OPENWEATHERMAP_API_KEY = os.getenv("OPENWEATHERMAP_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

assert GOOGLE_API_KEY, "GOOGLE_API_KEY is missing. Add it to your .env file."

print("GOOGLE_API_KEY loaded:", bool(GOOGLE_API_KEY))
print("OPENWEATHERMAP_API_KEY loaded:", bool(OPENWEATHERMAP_API_KEY))
print("TAVILY_API_KEY loaded:", bool(TAVILY_API_KEY))

GOOGLE_API_KEY loaded: True
OPENWEATHERMAP_API_KEY loaded: True
TAVILY_API_KEY loaded: True


## 1. ADK 2.0 graph-based workflow mental model

In LangGraph, you usually think in terms of:

```text
State → Node → Edge → Conditional Edge → State Update
```

In ADK 2.0 graph-based workflows, the comparable mental model is:

```text
Event → Node → Edge / Route → Event output, message, or state
```

Core objects:

| Concept | ADK 2.0 role |
|---|---|
| `Agent` | LLM-powered node |
| Python function | Deterministic code node |
| Tool function | Callable function used by an agent |
| `Workflow` | Graph container |
| `Event` | Data passed between workflow nodes |
| `RequestInput` | Human-in-the-loop pause/input node |

A simple graph route looks like:

```python
root_agent = Workflow(
    name="root_agent",
    edges=[
        ("START", node_1, node_2, node_3)
    ],
)
```

A branching graph looks like:

```python
root_agent = Workflow(
    name="router_workflow",
    edges=[
        ("START", classifier, router),
        (router, {
            "finance": finance_agent,
            "weather": weather_agent,
            "research": research_agent,
        })
    ],
)
```

In [3]:
# Imports used across the tutorial.
# If the ADK 2.0 beta API changes, check the import paths in the official ADK docs.

from google.adk import Agent, Workflow, Event
from pydantic import BaseModel, Field

/home/zadmin/Desktop/B7_GAAP_GCP/genai/lib/python3.12/site-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


In [4]:
def pretty_json(data: Any) -> str:
    return json.dumps(data, indent=2, ensure_ascii=False, default=str)

def compact_text(text: str, max_chars: int = 4000) -> str:
    if not text:
        return ""
    text = " ".join(str(text).split())
    return text[:max_chars] + ("..." if len(text) > max_chars else "")

# Part A — Single Agent

This is the simplest ADK pattern: one Gemini-powered agent, no external tools, no graph branching.

Use this when:
- the task is mostly reasoning or writing,
- no external API is required,
- there is no complex orchestration.

In [25]:
single_agent = Agent(
    name="single_agent",
    model="gemini-flash-latest",
    instruction="""
You are a concise AI assistant.
Answer the user's question clearly.
When useful, structure the answer into headings and steps.
""",
    output_schema=str,
)

root_agent = Workflow(
    name="single_agent_workflow",
    edges=[
        ("START", single_agent)
    ],
)

root_agent

Workflow(name='single_agent_workflow', description='', rerun_on_resume=True, wait_for_output=False, retry_config=None, timeout=None, input_schema=None, output_schema=None, state_schema=None, edges=[('START', LlmAgent(name='single_agent', description='', rerun_on_resume=False, wait_for_output=False, retry_config=None, timeout=None, input_schema=None, output_schema=<class 'str'>, state_schema=None, parent_agent=None, sub_agents=[], before_agent_callback=None, after_agent_callback=None, model='gemini-flash-latest', instruction="\nYou are a concise AI assistant.\nAnswer the user's question clearly.\nWhen useful, structure the answer into headings and steps.\n", global_instruction='', static_instruction=None, tools=[], generate_content_config=None, mode=None, parallel_worker=None, disallow_transfer_to_parent=False, disallow_transfer_to_peers=False, include_contents='default', output_key=None, planner=None, code_executor=None, before_model_callback=None, after_model_callback=None, on_model_e

In [28]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService

session_service = InMemorySessionService()
await session_service.create_session(app_name="SingleAgent",user_id="11",session_id="34534")

runner = Runner(agent = root_agent, app_name="SingleAgent",session_service=session_service)

from google.genai import types

events = runner.run_async(user_id="11",session_id="34534",new_message=types.Content(role='user',parts=[types.Part(text="Explain Quantum Computing")]))


In [29]:
async for event in events:

    print(event.content.parts[0].text)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


"QUANTUM COMPUTING OVERVIEW. 1. Definition: Quantum computing utilizes the principles of quantum mechanics to perform calculations that are significantly faster than classical computers for specific, complex problems. 2. Key Mechanics: Qubits: The fundamental unit of quantum information, which can represent 0, 1, or both states simultaneously. Superposition: A state that allows qubits to exist in multiple configurations at once, enabling parallel processing. Entanglement: A phenomenon where qubits become interconnected, meaning the state of one instantly influences the state of another regardless of distance. 3. Operational Steps: Step 1: Input data is encoded into a set of qubits. Step 2: Quantum gates manipulate the qubits to perform calculations through superposition and entanglement. Step 3: Quantum interference is used to amplify correct answers and cancel out incorrect ones. Step 4: Measuring the qubits collapses their state into a classical result. 4. Applications: Key use cases

In [43]:
import random
async def ask_adk_agent(agent, query:str):
    uid = str(random.randint(100,999))
    sid = str(random.randint(100,999))
    await session_service.create_session(app_name="Myagent",user_id=uid,session_id=sid)
    runner = Runner(agent = agent, app_name="Myagent",session_service=session_service)

    usermsg = types.Content(role='user',parts=[types.Part(text=query)])
    async for event in runner.run_async(user_id=uid,session_id=sid,new_message=usermsg):
        final_response = event.content.parts[0].text
        print(final_response)
    

In [44]:
await ask_adk_agent(root_agent,"what is AI?")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


"AI (Artificial Intelligence) is the field of computer science dedicated to creating systems capable of performing tasks that typically require human intelligence. These tasks include reasoning, learning from experience, and understanding language. ### Core Components: 1. Machine Learning: The process of training algorithms to find patterns in data. 2. Neural Networks: Systems modeled after the human brain to process complex information. 3. Natural Language Processing: The ability of machines to interpret and generate human speech or text. ### Common Categories: - Narrow AI: Specialized systems designed for specific tasks, such as virtual assistants or recommendation engines. - General AI: A theoretical form of intelligence where a machine can understand or learn any intellectual task that a human can."


# Part B — Tools

The next cells define the requested tools:

1. OpenWeatherMap
2. Tavily Search
3. Wikipedia
4. Yahoo Finance News
5. ChromaDB Vector Store helpers

These tools are normal Python functions. They can be attached to ADK agents.

In [30]:
import requests

def get_current_weather(city: str, country_code: Optional[str] = None) -> str:
    """
    Get current weather from OpenWeatherMap.

    Args:
        city: City name, for example 'Bengaluru'
        country_code: Optional ISO country code, for example 'IN'

    Returns:
        JSON string with weather information.
    """
    if not OPENWEATHERMAP_API_KEY:
        return "OPENWEATHERMAP_API_KEY is missing in .env. Add it to use the weather tool."

    location = city if not country_code else f"{city},{country_code}"
    url = "https://api.openweathermap.org/data/2.5/weather"
    params = {
        "q": location,
        "appid": OPENWEATHERMAP_API_KEY,
        "units": "metric",
    }

    try:
        response = requests.get(url, params=params, timeout=20)
        response.raise_for_status()
        data = response.json()
        result = {
            "city": data.get("name"),
            "country": data.get("sys", {}).get("country"),
            "temperature_celsius": data.get("main", {}).get("temp"),
            "feels_like_celsius": data.get("main", {}).get("feels_like"),
            "humidity": data.get("main", {}).get("humidity"),
            "weather": data.get("weather", [{}])[0].get("description"),
            "wind_speed": data.get("wind", {}).get("speed"),
        }
        return pretty_json(result)
    except Exception as exc:
        return f"Weather tool error: {exc}"

In [31]:
def tavily_search(query: str, max_results: int = 5) -> str:
    """
    Search the web using Tavily.

    Args:
        query: Search query.
        max_results: Number of search results.

    Returns:
        JSON string with search results.
    """
    if not TAVILY_API_KEY:
        return "TAVILY_API_KEY is missing in .env. Add it to use Tavily search."

    try:
        from tavily import TavilyClient
        client = TavilyClient(api_key=TAVILY_API_KEY)
        response = client.search(
            query=query,
            max_results=max_results,
            include_answer=True,
            include_raw_content=False,
        )
        results = {
            "answer": response.get("answer"),
            "results": [
                {
                    "title": item.get("title"),
                    "url": item.get("url"),
                    "content": compact_text(item.get("content", ""), 700),
                }
                for item in response.get("results", [])
            ],
        }
        return pretty_json(results)
    except Exception as exc:
        return f"Tavily search tool error: {exc}"

In [32]:
def wikipedia_summary(topic: str, sentences: int = 5) -> str:
    """
    Fetch a summary from Wikipedia.

    Args:
        topic: Topic or page title.
        sentences: Number of sentences in the summary.

    Returns:
        Summary string.
    """
    try:
        import wikipedia
        wikipedia.set_lang("en")
        return wikipedia.summary(topic, sentences=sentences, auto_suggest=True, redirect=True)
    except Exception as exc:
        return f"Wikipedia tool error: {exc}"

In [33]:
def yahoo_finance_news(ticker: str, max_items: int = 5) -> str:
    """
    Get recent Yahoo Finance news for a stock ticker.

    Args:
        ticker: Stock ticker, for example 'AAPL', 'MSFT', 'GOOGL'.
        max_items: Maximum news items to return.

    Returns:
        JSON string with news headlines and links.
    """
    try:
        import yfinance as yf
        stock = yf.Ticker(ticker)
        news = stock.news or []
        cleaned = []
        for item in news[:max_items]:
            content = item.get("content", item)
            cleaned.append({
                "title": content.get("title") or item.get("title"),
                "publisher": content.get("provider", {}).get("displayName") if isinstance(content.get("provider"), dict) else item.get("publisher"),
                "summary": compact_text(content.get("summary") or item.get("summary", ""), 700),
                "url": content.get("canonicalUrl", {}).get("url") if isinstance(content.get("canonicalUrl"), dict) else item.get("link"),
                "published": content.get("pubDate") or item.get("providerPublishTime"),
            })
        return pretty_json(cleaned)
    except Exception as exc:
        return f"Yahoo Finance news tool error: {exc}"

# Part C — Single Agent with Tools

In this pattern, one agent has access to multiple tools and decides when to call them.

Use this when:
- one agent can own the whole task,
- tool selection is simple,
- routing does not need to be deterministic.

In [34]:
tool_enabled_agent = Agent(
    name="tool_enabled_agent",
    model="gemini-flash-latest",
    instruction="""
You are a tool-using assistant.

Use tools when needed:
- Use get_current_weather for current weather.
- Use tavily_search for fresh web information.
- Use wikipedia_summary for stable encyclopedia-style background.
- Use yahoo_finance_news for recent company/stock news.

When you use a tool, cite the tool result in your explanation.
If a required API key is missing, explain which key is missing and how to add it.
""",
    tools=[
        get_current_weather,
        tavily_search,
        wikipedia_summary,
        yahoo_finance_news,
    ],
    output_schema=str,
)

tool_enabled_agent_workflow = Workflow(
    name="tool_enabled_agent_workflow",
    edges=[
        ("START", tool_enabled_agent)
    ],
)

tool_enabled_agent_workflow

Workflow(name='tool_enabled_agent_workflow', description='', rerun_on_resume=True, wait_for_output=False, retry_config=None, timeout=None, input_schema=None, output_schema=None, state_schema=None, edges=[('START', LlmAgent(name='tool_enabled_agent', description='', rerun_on_resume=False, wait_for_output=False, retry_config=None, timeout=None, input_schema=None, output_schema=<class 'str'>, state_schema=None, parent_agent=None, sub_agents=[], before_agent_callback=None, after_agent_callback=None, model='gemini-flash-latest', instruction='\nYou are a tool-using assistant.\n\nUse tools when needed:\n- Use get_current_weather for current weather.\n- Use tavily_search for fresh web information.\n- Use wikipedia_summary for stable encyclopedia-style background.\n- Use yahoo_finance_news for recent company/stock news.\n\nWhen you use a tool, cite the tool result in your explanation.\nIf a required API key is missing, explain which key is missing and how to add it.\n', global_instruction='', s

In [46]:
await ask_adk_agent(tool_enabled_agent_workflow,"What is current weather in Delhi Today")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


None
None
None
None
"Based on the current weather data for Delhi, India, the temperature is 33.05°C (feeling like 32.87°C) with hazy conditions. The humidity is at 35% and there is a light wind of 1.54 m/s. (Source: get_current_weather)"


In [47]:
await ask_adk_agent(tool_enabled_agent_workflow,"Who is current president of USA")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


None


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


None
None
None
"The 47th and current president of the United States is Donald Trump. He was sworn into office on January 20, 2025, and his term is scheduled to end on January 20, 2029 (Source: tavily_search result)."


## Example prompts for the tool-enabled agent

```text
What is the current weather in Bengaluru?
```

```text
Search recent developments in Google ADK graph workflows and summarize in 5 bullets.
```

```text
Give me a beginner-friendly explanation of retrieval augmented generation using Wikipedia.
```

```text
Get recent news for MSFT and summarize potential business implications.
```

# Part D — Single Agent with Tools and HITL

Human-in-the-loop is useful when the agent needs approval before performing or finalizing an action.

In this example:

1. The agent prepares an investment/news briefing.
2. A human reviews the draft.
3. The workflow either revises or finalizes the answer.

ADK graph-based workflows support HITL using `RequestInput`.

In [48]:
from google.adk.events import RequestInput

class HumanReview(BaseModel):
    approved: bool = Field(description="Whether the human reviewer approves the draft.")
    feedback: str = Field(default="", description="Required when approved is false.")

def request_human_review(node_input: str):
    """
    Pause workflow and ask a human to approve or provide feedback.
    """
    yield RequestInput(
        message="""Review the agent draft. Reply with structured approval:
{
  "approved": true,
  "feedback": ""
}

or

{
  "approved": false,
  "feedback": "Explain what should be changed."
}
""",
        payload={"draft": node_input},
        response_schema=HumanReview,
    )

def hitl_router(review: HumanReview) -> str:
    """
    Route based on human approval.
    """
    if review.approved:
        return "approved"
    return "revise"

draft_with_tools_agent = Agent(
    name="draft_with_tools_agent",
    model="gemini-flash-latest",
    instruction="""
Create a useful draft answer for the user. Use tools if required.
For finance questions, use yahoo_finance_news.
For fresh web questions, use tavily_search.
For background explanation, use wikipedia_summary.
Return a clear draft for human review.
""",
    tools=[tavily_search, wikipedia_summary, yahoo_finance_news],
    output_schema=str,
)

revision_agent = Agent(
    name="revision_agent",
    model="gemini-flash-latest",
    instruction="""
Revise the draft using the human feedback.
Preserve accurate tool-derived facts.
Improve clarity, structure, and completeness.
""",
    output_schema=str,
)

finalize_agent = Agent(
    name="finalize_agent",
    model="gemini-flash-latest",
    instruction="""
Finalize the approved answer.
Make it concise, structured, and ready to send.
""",
    output_schema=str,
)

hitl_tool_workflow = Workflow(
    name="hitl_tool_workflow",
    edges=[
        ("START", draft_with_tools_agent, request_human_review, hitl_router),
        (hitl_router, {
            "approved": finalize_agent,
            "revise": revision_agent,
        }),
    ],
)

hitl_tool_workflow

Workflow(name='hitl_tool_workflow', description='', rerun_on_resume=True, wait_for_output=False, retry_config=None, timeout=None, input_schema=None, output_schema=None, state_schema=None, edges=[('START', LlmAgent(name='draft_with_tools_agent', description='', rerun_on_resume=False, wait_for_output=False, retry_config=None, timeout=None, input_schema=None, output_schema=<class 'str'>, state_schema=None, parent_agent=None, sub_agents=[], before_agent_callback=None, after_agent_callback=None, model='gemini-flash-latest', instruction='\nCreate a useful draft answer for the user. Use tools if required.\nFor finance questions, use yahoo_finance_news.\nFor fresh web questions, use tavily_search.\nFor background explanation, use wikipedia_summary.\nReturn a clear draft for human review.\n', global_instruction='', static_instruction=None, tools=[<function tavily_search at 0x76e772a44f40>, <function wikipedia_summary at 0x76e772a44ae0>, <function yahoo_finance_news at 0x76e772a45d00>], generate

In [49]:
await ask_adk_agent(hitl_tool_workflow,"Write an article on Agentic AI in 2 lines")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


None


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


None
None
None
"Based on the provided information, **AI agents** (also known as agentic AI or compound AI systems) are intelligent systems designed to pursue specific goals by using tools and taking actions autonomously within a set of human-defined constraints.\n\nHere is a breakdown of their key characteristics and components:\n\n### Key Attributes:\n*   **Goal-Directed Behavior:** They don't just respond to prompts; they work towards achieving a defined objective.\n*   **Natural Language Interfaces:** They can communicate and receive instructions through human language.\n*   **Tool Usage:** They have the capacity to use external tools and software to complete tasks.\n*   **Multi-Step Reasoning:** They can break down complex objectives into smaller, actionable steps.\n\n### How They Work:\nThe \"brain\" of an AI agent is typically a **Large Language Model (LLM)**, which drives the control flow. To function effectively, these systems often integrate:\n*   **Memory:** To keep track of 

## HITL design notes

Use HITL for:

- approval before sending customer communication,
- validation before financial or operational recommendation,
- checking generated code before deployment,
- confirming sensitive decisions,
- collecting missing business inputs.

In a real application, the human response would come from a UI, API, workflow platform, or chat interface.

# Part E — Multi-Agent Supervisor Pattern

The supervisor pattern uses one coordinator/supervisor agent to decide which specialist should handle the task.

In this tutorial, we create:

- `supervisor_router_agent`
- `weather_specialist_agent`
- `research_specialist_agent`
- `finance_specialist_agent`
- `general_response_agent`

The graph structure:

```text
START
  ↓
Supervisor Router
  ↓
weather / research / finance / general
  ↓
Specialist Agent
  ↓
Final Response
```

In [ ]:
class RouteDecision(BaseModel):
    route: str = Field(description="One of: weather, research, finance, general")
    reason: str = Field(description="Brief reason for the routing decision")

supervisor_router_agent = Agent(
    name="supervisor_router_agent",
    model="gemini-flash-latest",
    instruction="""
You are a supervisor router.
Classify the user's request into exactly one route:

- weather: current weather, forecast, climate for a city
- research: fresh web search, current topics, market trends, latest information
- finance: stock, company news, market news, ticker-specific news
- general: general reasoning, writing, explanation, or anything else

Return only structured output matching the schema.
""",
    output_schema=RouteDecision,
)

def route_from_supervisor(decision: RouteDecision) -> str:
    allowed = {"weather", "research", "finance", "general"}
    route = decision.route.lower().strip()
    return route if route in allowed else "general"

weather_specialist_agent = Agent(
    name="weather_specialist_agent",
    model="gemini-flash-latest",
    instruction="""
You are a weather specialist.
Use get_current_weather when the user asks for current weather.
Explain the result in practical language.
""",
    tools=[get_current_weather],
    output_schema=str,
)

research_specialist_agent = Agent(
    name="research_specialist_agent",
    model="gemini-flash-latest",
    instruction="""
You are a research specialist.
Use Tavily for fresh web information and Wikipedia for background context.
Summarize findings in a structured way.
""",
    tools=[tavily_search, wikipedia_summary],
    output_schema=str,
)

finance_specialist_agent = Agent(
    name="finance_specialist_agent",
    model="gemini-flash-latest",
    instruction="""
You are a finance news specialist.
Use yahoo_finance_news for ticker-specific news.
Do not provide investment advice. Provide informational analysis only.
""",
    tools=[yahoo_finance_news, tavily_search],
    output_schema=str,
)

general_response_agent = Agent(
    name="general_response_agent",
    model="gemini-flash-latest",
    instruction="""
You are a general assistant.
Answer clearly and directly.
""",
    output_schema=str,
)

supervisor_workflow = Workflow(
    name="supervisor_workflow",
    edges=[
        ("START", supervisor_router_agent, route_from_supervisor),
        (route_from_supervisor, {
            "weather": weather_specialist_agent,
            "research": research_specialist_agent,
            "finance": finance_specialist_agent,
            "general": general_response_agent,
        }),
    ],
)

supervisor_workflow

## Supervisor pattern variations

You can extend the supervisor pattern in three ways:

1. **Supervisor as router only**  
   The supervisor only chooses the specialist.

2. **Supervisor as planner**  
   The supervisor creates a plan and assigns sub-tasks.

3. **Supervisor as reviewer**  
   The supervisor reviews specialist outputs and asks for revision if needed.

For enterprise projects, the third option is usually stronger because it adds quality control.

# Part F — RAG with ChromaDB

RAG flow:

```text
User Query
  ↓
Retrieve relevant chunks from ChromaDB
  ↓
Send query + retrieved context to Gemini agent
  ↓
Answer grounded in retrieved context
```

This tutorial uses:

- ChromaDB as vector store
- Gemini embeddings through `GOOGLE_API_KEY`
- Gemini LLM through ADK Agent

In [ ]:
import chromadb
from chromadb.api.types import EmbeddingFunction, Documents, Embeddings
from google import genai

class GeminiEmbeddingFunction(EmbeddingFunction):
    """
    ChromaDB embedding function using Gemini embeddings.
    """
    def __init__(self, model_name: str = "text-embedding-004"):
        self.client = genai.Client(api_key=GOOGLE_API_KEY)
        self.model_name = model_name

    def __call__(self, input: Documents) -> Embeddings:
        embeddings = []
        for text in input:
            response = self.client.models.embed_content(
                model=self.model_name,
                contents=text,
            )
            vector = response.embeddings[0].values
            embeddings.append(vector)
        return embeddings

CHROMA_PATH = "./chroma_adk_tutorial_db"
COLLECTION_NAME = "company_knowledge"

embedding_fn = GeminiEmbeddingFunction()

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_fn,
)

print("Chroma collection ready:", COLLECTION_NAME)

In [ ]:
sample_docs = [
    {
        "id": "doc_001",
        "text": "Google ADK is an agent development framework for building agents, tools, workflows, and multi-agent systems. ADK 2.0 introduces graph-based workflows for deterministic routing.",
        "source": "sample_adk_notes",
    },
    {
        "id": "doc_002",
        "text": "Retrieval Augmented Generation, or RAG, improves LLM responses by retrieving external context from a knowledge base before generating an answer.",
        "source": "sample_rag_notes",
    },
    {
        "id": "doc_003",
        "text": "Corrective RAG adds a validation step after retrieval. If retrieved context is weak, the system rewrites the query or performs external search before answering.",
        "source": "sample_corrective_rag_notes",
    },
    {
        "id": "doc_004",
        "text": "Human-in-the-loop workflows ask a human for approval, missing data, or validation before continuing execution.",
        "source": "sample_hitl_notes",
    },
]

existing = collection.get(ids=[doc["id"] for doc in sample_docs])
existing_ids = set(existing.get("ids", []))

for doc in sample_docs:
    if doc["id"] not in existing_ids:
        collection.add(
            ids=[doc["id"]],
            documents=[doc["text"]],
            metadatas=[{"source": doc["source"]}],
        )

print("Documents in collection:", collection.count())

In [ ]:
def retrieve_from_chroma(query: str, n_results: int = 4) -> str:
    """
    Retrieve relevant chunks from ChromaDB.

    Args:
        query: User question or search query.
        n_results: Number of chunks to retrieve.

    Returns:
        JSON string containing retrieved chunks and metadata.
    """
    try:
        results = collection.query(
            query_texts=[query],
            n_results=n_results,
            include=["documents", "metadatas", "distances"],
        )
        retrieved = []
        docs = results.get("documents", [[]])[0]
        metas = results.get("metadatas", [[]])[0]
        distances = results.get("distances", [[]])[0]

        for i, doc in enumerate(docs):
            retrieved.append({
                "rank": i + 1,
                "text": doc,
                "metadata": metas[i] if i < len(metas) else {},
                "distance": distances[i] if i < len(distances) else None,
            })
        return pretty_json(retrieved)
    except Exception as exc:
        return f"Chroma retrieval tool error: {exc}"

## RAG agent

This is the simplest RAG implementation: one agent with a retrieval tool.

In [ ]:
rag_agent = Agent(
    name="rag_agent",
    model="gemini-flash-latest",
    instruction="""
You are a RAG assistant.

Process:
1. Use retrieve_from_chroma to retrieve relevant context.
2. Answer using only the retrieved context when possible.
3. If retrieved context is insufficient, clearly say what is missing.
4. Cite sources from the retrieved metadata when available.

Do not invent facts that are not supported by the retrieved context.
""",
    tools=[retrieve_from_chroma],
    output_schema=str,
)

rag_workflow = Workflow(
    name="rag_workflow",
    edges=[
        ("START", rag_agent)
    ],
)

rag_workflow

## Example RAG prompts

```text
What is Corrective RAG?
```

```text
How does ADK 2.0 help with deterministic workflows?
```

```text
Explain HITL in agentic workflows.
```

# Part G — Graph-Based RAG

A more controlled graph-based RAG approach separates retrieval and generation into different nodes.

Graph:

```text
START
  ↓
Retrieve Context
  ↓
Generate Answer
  ↓
Completed Message
```

In [ ]:
class RetrievedContext(BaseModel):
    query: str
    context: List[Dict[str, Any]]

def retrieve_context_node(user_query: str) -> Event:
    raw = retrieve_from_chroma(user_query, n_results=4)
    try:
        context = json.loads(raw)
    except Exception:
        context = [{"text": raw, "metadata": {}, "distance": None}]
    return Event(output=RetrievedContext(query=user_query, context=context))

graph_rag_answer_agent = Agent(
    name="graph_rag_answer_agent",
    model="gemini-flash-latest",
    input_schema=RetrievedContext,
    instruction="""
Answer the user's query using the retrieved context.

User query:
{RetrievedContext.query}

Retrieved context:
{RetrievedContext.context}

Rules:
- Use only the retrieved context when possible.
- If context is insufficient, say so.
- Include source metadata if available.
""",
    output_schema=str,
)

def completed_node(answer: str) -> Event:
    return Event(message=answer, output=answer)

graph_rag_workflow = Workflow(
    name="graph_rag_workflow",
    edges=[
        ("START", retrieve_context_node, graph_rag_answer_agent, completed_node)
    ],
)

graph_rag_workflow

# Part H — Corrective RAG Using ADK Graph-Based Workflow

Corrective RAG improves basic RAG by checking whether retrieval quality is good enough.

Graph:

```text
START
  ↓
Retrieve from Chroma
  ↓
Grade Retrieval Quality
  ↓
 ┌────────────── good ──────────────┐
 ↓                                  ↓
Generate grounded answer       Query rewrite / web search
                                      ↓
                               Generate corrected answer
```

Correction strategies:

1. **Rewrite the query** and retrieve again.
2. **Use web search** if vector context is insufficient.
3. **Ask human for missing context** in high-risk scenarios.
4. **Refuse/qualify the answer** when sources are inadequate.

In [ ]:
class RetrievalGrade(BaseModel):
    grade: str = Field(description="One of: good, weak")
    reason: str = Field(description="Reason for the grade")
    improved_query: Optional[str] = Field(default=None, description="Improved query if retrieval is weak")

def grade_retrieval_quality_node(retrieved: RetrievedContext) -> RetrievalGrade:
    """
    Deterministic heuristic retrieval grader.

    In production, you can replace this with an LLM grader agent.
    """
    if not retrieved.context:
        return RetrievalGrade(
            grade="weak",
            reason="No retrieved context.",
            improved_query=retrieved.query,
        )

    distances = [
        item.get("distance")
        for item in retrieved.context
        if isinstance(item.get("distance"), (int, float))
    ]

    combined_text = " ".join([item.get("text", "") for item in retrieved.context])

    if len(combined_text.strip()) < 100:
        return RetrievalGrade(
            grade="weak",
            reason="Retrieved context is too short.",
            improved_query=retrieved.query,
        )

    if distances and min(distances) > 1.2:
        return RetrievalGrade(
            grade="weak",
            reason=f"Best vector distance is high: {min(distances):.3f}",
            improved_query=retrieved.query,
        )

    return RetrievalGrade(
        grade="good",
        reason="Retrieved context appears sufficient.",
        improved_query=None,
    )

def corrective_rag_router(grade: RetrievalGrade) -> str:
    return "good" if grade.grade.lower().strip() == "good" else "weak"

In [ ]:
class CorrectedContext(BaseModel):
    original_query: str
    retrieval_grade: RetrievalGrade
    vector_context: List[Dict[str, Any]]
    web_context: Optional[str] = None

def web_search_correction_node(grade: RetrievalGrade) -> Event:
    """
    Use Tavily as fallback if retrieval quality is weak.
    """
    query = grade.improved_query or "agentic AI retrieval augmented generation"
    web_result = tavily_search(query=query, max_results=5)

    corrected = CorrectedContext(
        original_query=query,
        retrieval_grade=grade,
        vector_context=[],
        web_context=web_result,
    )
    return Event(output=corrected)

corrective_rag_good_answer_agent = Agent(
    name="corrective_rag_good_answer_agent",
    model="gemini-flash-latest",
    input_schema=RetrievedContext,
    instruction="""
You are answering with strong vector-store context.

Use this retrieved context:
{RetrievedContext.context}

Answer the original query:
{RetrievedContext.query}

Rules:
- Ground the answer in context.
- Mention source metadata where available.
- Do not overclaim.
""",
    output_schema=str,
)

corrective_rag_web_answer_agent = Agent(
    name="corrective_rag_web_answer_agent",
    model="gemini-flash-latest",
    input_schema=CorrectedContext,
    instruction="""
The vector database retrieval was weak.

Use the fallback web context below to answer:
{CorrectedContext.web_context}

Retrieval grade:
{CorrectedContext.retrieval_grade}

Rules:
- Explain that fallback search was used because local retrieval was weak.
- Summarize the answer clearly.
- Do not invent facts beyond the provided context.
""",
    output_schema=str,
)

In [ ]:
corrective_rag_workflow = Workflow(
    name="corrective_rag_workflow",
    edges=[
        ("START", retrieve_context_node, grade_retrieval_quality_node, corrective_rag_router),
        (corrective_rag_router, {
            "good": corrective_rag_good_answer_agent,
            "weak": web_search_correction_node,
        }),
        (web_search_correction_node, corrective_rag_web_answer_agent),
    ],
)

corrective_rag_workflow

## More production-grade Corrective RAG design

A production version should include:

1. **Retrieval evaluator**
   - relevance
   - coverage
   - source credibility
   - freshness
   - similarity score

2. **Query rewriting**
   - expand acronyms
   - add domain terms
   - decompose complex questions

3. **Second retrieval pass**
   - retrieve again with rewritten query

4. **Fallback search**
   - Tavily or enterprise search

5. **Human review**
   - for legal, finance, medical, compliance, or customer-facing content

6. **Final answer validation**
   - answer must cite retrieved sources
   - unsupported claims should be removed

# Part I — Corrective RAG with HITL

For high-risk use cases, do not rely only on automated correction.

Graph idea:

```text
Retrieve
  ↓
Grade
  ↓
Weak?
  ↓
Ask human for missing document / approval
  ↓
Answer
```

Below is a reusable HITL node that can be inserted into the weak-retrieval path.

In [ ]:
class MissingContextHumanInput(BaseModel):
    additional_context: str = Field(description="Additional context supplied by a human.")
    allow_web_fallback: bool = Field(description="Whether web fallback is allowed.")

def request_missing_context_node(grade: RetrievalGrade):
    yield RequestInput(
        message="""The vector database retrieval appears weak.
Please provide additional context or allow web fallback.

Respond as:
{
  "additional_context": "Paste relevant context here, or leave empty.",
  "allow_web_fallback": true
}
""",
        payload={"retrieval_grade": grade.model_dump()},
        response_schema=MissingContextHumanInput,
    )

def human_context_router(human_input: MissingContextHumanInput) -> str:
    if human_input.additional_context and human_input.additional_context.strip():
        return "human_context"
    if human_input.allow_web_fallback:
        return "web_fallback"
    return "insufficient_context"

class HumanProvidedContext(BaseModel):
    context: str

def convert_human_context_node(human_input: MissingContextHumanInput) -> Event:
    return Event(output=HumanProvidedContext(context=human_input.additional_context))

human_context_answer_agent = Agent(
    name="human_context_answer_agent",
    model="gemini-flash-latest",
    input_schema=HumanProvidedContext,
    instruction="""
Answer using the human-provided context below.

Context:
{HumanProvidedContext.context}

Rules:
- Use only the human-provided context.
- If it is insufficient, say what is missing.
""",
    output_schema=str,
)

insufficient_context_agent = Agent(
    name="insufficient_context_agent",
    model="gemini-flash-latest",
    instruction="""
Explain that the system cannot answer confidently because both retrieved context and approved fallback context are insufficient.
Ask for the specific missing document, data source, or clarification.
""",
    output_schema=str,
)

corrective_rag_hitl_workflow = Workflow(
    name="corrective_rag_hitl_workflow",
    edges=[
        ("START", retrieve_context_node, grade_retrieval_quality_node, corrective_rag_router),
        (corrective_rag_router, {
            "good": corrective_rag_good_answer_agent,
            "weak": request_missing_context_node,
        }),
        (request_missing_context_node, human_context_router),
        (human_context_router, {
            "human_context": convert_human_context_node,
            "web_fallback": web_search_correction_node,
            "insufficient_context": insufficient_context_agent,
        }),
        (convert_human_context_node, human_context_answer_agent),
        (web_search_correction_node, corrective_rag_web_answer_agent),
    ],
)

corrective_rag_hitl_workflow

# Part J — How to run these workflows

## Option 1: ADK Web UI

Create a project folder and save one workflow as `agent.py`.

Example structure:

```text
adk_graph_tutorial/
  .env
  agent.py
```

Then run:

```bash
adk web
```

## Option 2: ADK CLI

```bash
adk run adk_graph_tutorial
```

## Option 3: Notebook-driven development

Use this notebook for:
- building agents,
- validating tool functions,
- prototyping graph routes,
- copying stable workflows into `agent.py`.

In [ ]:
from pathlib import Path

project_dir = Path("adk_graph_tutorial_project")
project_dir.mkdir(exist_ok=True)

agent_py = r"""
import os
import json
from typing import Any, Dict, List
from dotenv import load_dotenv
from pydantic import BaseModel
from google.adk import Agent, Workflow, Event

load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

def pretty_json(data):
    return json.dumps(data, indent=2, ensure_ascii=False, default=str)

def demo_retrieve_from_chroma(query: str, n_results: int = 4) -> str:
    # Replace this demo function with your real ChromaDB retrieval code.
    return pretty_json([
        {
            "rank": 1,
            "text": "RAG retrieves external context before generating an answer.",
            "metadata": {"source": "demo"},
            "distance": 0.3,
        }
    ])

class RetrievedContext(BaseModel):
    query: str
    context: List[Dict[str, Any]]

def retrieve_context_node(user_query: str) -> Event:
    raw = demo_retrieve_from_chroma(user_query)
    context = json.loads(raw)
    return Event(output=RetrievedContext(query=user_query, context=context))

graph_rag_answer_agent = Agent(
    name="graph_rag_answer_agent",
    model="gemini-flash-latest",
    input_schema=RetrievedContext,
    instruction=\"\"\"
Answer the user's query using the retrieved context.

User query:
{RetrievedContext.query}

Retrieved context:
{RetrievedContext.context}

Rules:
- Use only the retrieved context when possible.
- If context is insufficient, say so.
- Include source metadata if available.
\"\"\",
    output_schema=str,
)

def completed_node(answer: str) -> Event:
    return Event(message=answer, output=answer)

root_agent = Workflow(
    name="graph_rag_workflow",
    edges=[
        ("START", retrieve_context_node, graph_rag_answer_agent, completed_node)
    ],
)
"""

(project_dir / "agent.py").write_text(agent_py, encoding="utf-8")
(project_dir / "__init__.py").write_text("", encoding="utf-8")

print(f"Created: {project_dir / 'agent.py'}")
print("Run with: adk web")

# Part K — Practice exercises

## Exercise 1 — Single Agent

Create a single agent that explains technical concepts to non-technical business stakeholders.

## Exercise 2 — Tool Agent

Create a tool agent that can:
- get current weather,
- search latest web information,
- summarize a Wikipedia concept.

## Exercise 3 — HITL

Add human approval before the agent finalizes a customer email.

## Exercise 4 — Supervisor Pattern

Create a supervisor with four specialists:
- Sales proposal specialist
- Technical architect
- Risk reviewer
- Pricing analyst

## Exercise 5 — RAG

Load your company documents into ChromaDB and build a RAG agent.

## Exercise 6 — Corrective RAG

Add retrieval grading and a fallback path:
- rewrite query,
- retrieve again,
- web search,
- human input if still weak.

# Summary

You have created six progressively advanced ADK patterns:

| Pattern | Best use case |
|---|---|
| Single agent | Simple reasoning/writing |
| Single agent with tools | One assistant with external API access |
| Tool agent + HITL | Approval-based flows |
| Multi-agent supervisor | Specialist delegation |
| RAG | Grounded knowledge-base answering |
| Corrective RAG | Robust retrieval with fallback and validation |

The key shift in ADK 2.0 is that you can now model these systems as **graph-based workflows**, closer to LangGraph-style orchestration, while staying inside the Google/Gemini ADK ecosystem.